# Laboratorio 8 — MM3014 Teoría de Probabilidades
## Etapas 3 y 4: Presupuesto, costo e intercambio de repetidas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

rng = np.random.default_rng(2026)

---
## Etapa 3: Incorporación del presupuesto y costo

**Parámetros:** N = 100 estampas, S = 7 por sobre, precio Q 9.50, presupuesto Q 1000, R = 10 000 simulaciones, semilla 2026.

In [ ]:
# ── Parámetros Etapa 3 ──────────────────────────────────────────────────────
N      = 100      # total de estampas
S      = 7        # estampas por sobre
PRECIO = 9.50     # Q por sobre
BUDGET = 1000.0   # presupuesto total Q
R      = 10_000   # simulaciones

# ── Simulación ───────────────────────────────────────────────────────────────
completado   = np.zeros(R, dtype=bool)
sobres_cmp   = np.zeros(R, dtype=int)
distintas_nc = []   # estampas distintas en simulaciones NO exitosas

for i in range(R):
    coleccion = np.zeros(N, dtype=bool)
    gasto     = 0.0
    sobres    = 0

    while gasto + PRECIO <= BUDGET and coleccion.sum() < N:
        stickers  = rng.integers(0, N, size=S)   # sobre con S estampas aleatorias
        coleccion[stickers] = True
        gasto  += PRECIO
        sobres += 1

    completado[i]   = coleccion.sum() == N
    sobres_cmp[i]   = sobres
    if not completado[i]:
        distintas_nc.append(coleccion.sum())

distintas_nc = np.array(distintas_nc)

# ── Resultados ───────────────────────────────────────────────────────────────
prob_exito      = completado.mean()
esp_sobres      = sobres_cmp.mean()
esp_distintas   = distintas_nc.mean() if len(distintas_nc) > 0 else np.nan

print(f"Probabilidad de completar el álbum:          {prob_exito:.4f}  ({prob_exito*100:.2f}%)")
print(f"Número esperado de sobres comprados:          {esp_sobres:.2f}")
print(f"Estampas distintas (casos no exitosos, E[X]): {esp_distintas:.2f}")
print(f"Simulaciones completadas:                     {completado.sum():,} / {R:,}")
print(f"Simulaciones no completadas:                  {(~completado).sum():,} / {R:,}")

In [ ]:
# ── Diagrama de barras: completó vs no completó ──────────────────────────────
fig, ax = plt.subplots(figsize=(6, 4))

categorias  = ["Completó", "No completó"]
proporciones = [prob_exito, 1 - prob_exito]
colores      = ["#2ecc71", "#e74c3c"]

bars = ax.bar(categorias, proporciones, color=colores, edgecolor="black", width=0.45)

for bar, prop in zip(bars, proporciones):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{prop*100:.2f}%",
            ha="center", va="bottom", fontsize=11, fontweight="bold")

ax.set_ylim(0, 1.15)
ax.set_ylabel("Proporción", fontsize=12)
ax.set_title("Etapa 3 — Resultado de 10 000 simulaciones\n(presupuesto Q 1000)", fontsize=12)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("etapa3_barras.png", dpi=150)
plt.show()

### Preguntas de análisis — Etapa 3

In [ ]:
# ── Pregunta 1 ───────────────────────────────────────────────────────────────
max_sobres     = int(BUDGET // PRECIO)          # máximo entero de sobres comprables
min_teo_sobres = int(np.ceil(N / S))            # mínimo teórico sin repetidos

print("=" * 55)
print("Pregunta 1")
print("=" * 55)
print(f"Máximo de sobres comprables con Q {BUDGET:.0f}: {max_sobres}")
print(f"  → Gasto: Q {max_sobres * PRECIO:.2f}")
print(f"  → Estampas máximas (sin rep.): {max_sobres * S}  (N = {N})")
print(f"Mínimo teórico de sobres (ceil(N/S)):  {min_teo_sobres}")
print()
if max_sobres * S >= N:
    print(f"Con {max_sobres} sobres se cubren {max_sobres*S} 'slots', "
          f"suficiente en teoría (ignorando repetidos).")
else:
    print("No sería suficiente ni en el caso sin repetidos.")
print(f"Sin embargo, por repetidos se necesitan en promedio ~{esp_sobres:.1f} sobres.")

In [ ]:
# ── Pregunta 2: simulación con caja de 104 sobres (Q 975) ───────────────────
COSTO_CAJA   = 975.0
SOBRES_CAJA  = 104

rng2 = np.random.default_rng(2026)   # misma semilla, simulación independiente

completado_caja = np.zeros(R, dtype=bool)

for i in range(R):
    coleccion = np.zeros(N, dtype=bool)
    for _ in range(SOBRES_CAJA):
        stickers = rng2.integers(0, N, size=S)
        coleccion[stickers] = True
    completado_caja[i] = coleccion.sum() == N

prob_caja = completado_caja.mean()

print("=" * 55)
print("Pregunta 2 — Caja de 104 sobres (Q 975)")
print("=" * 55)
print(f"P(completar) con caja:         {prob_caja:.4f}  ({prob_caja*100:.2f}%)")
print(f"P(completar) sobres sueltos:   {prob_exito:.4f}  ({prob_exito*100:.2f}%)")
diferencia = prob_caja - prob_exito
print(f"Diferencia (caja − sueltos):   {diferencia:+.4f}  ({diferencia*100:+.2f} pp)")
print()
if diferencia > 0:
    print("→ Conviene comprar la caja.")
elif diferencia < 0:
    print("→ Conviene comprar sobres sueltos.")
else:
    print("→ Es indiferente.")

In [ ]:
# ── Pregunta 3: estrategia mixta (caja + sueltos) ───────────────────────────
#
# Caja: Q 975, 104 sobres.  Presupuesto restante: Q 1000 − Q 975 = Q 25.
# Con Q 25 se pueden comprar floor(25 / 9.50) = 2 sobres sueltos adicionales.
# Total: 104 + 2 = 106 sobres, gasto total: 975 + 2*9.50 = Q 994.

presupuesto_restante = BUDGET - COSTO_CAJA
extra_sueltos = int(presupuesto_restante // PRECIO)
total_sobres_mixto = SOBRES_CAJA + extra_sueltos
costo_total_mixto  = COSTO_CAJA + extra_sueltos * PRECIO

rng3 = np.random.default_rng(2026)

completado_mixto = np.zeros(R, dtype=bool)

for i in range(R):
    coleccion = np.zeros(N, dtype=bool)
    for _ in range(total_sobres_mixto):
        stickers = rng3.integers(0, N, size=S)
        coleccion[stickers] = True
    completado_mixto[i] = coleccion.sum() == N

prob_mixto = completado_mixto.mean()

print("=" * 55)
print("Pregunta 3 — Estrategia mixta")
print("=" * 55)
print(f"Presupuesto restante tras la caja: Q {presupuesto_restante:.2f}")
print(f"Sobres sueltos adicionales:        {extra_sueltos}")
print(f"Total sobres (caja + sueltos):     {total_sobres_mixto}")
print(f"Costo total:                       Q {costo_total_mixto:.2f}  "
      f"(ahorro Q {BUDGET - costo_total_mixto:.2f})")
print()
print(f"P(completar) mixta:          {prob_mixto:.4f}  ({prob_mixto*100:.2f}%)")
print(f"P(completar) solo caja:      {prob_caja:.4f}  ({prob_caja*100:.2f}%)")
print(f"P(completar) sobres sueltos: {prob_exito:.4f}  ({prob_exito*100:.2f}%)")
print()
print("Estrategia recomendada: la que tenga mayor probabilidad de completar.")

---
## Etapa 4: Efecto del intercambio de repetidas

**Regla:** cada K estampas repetidas se canjean por 1 estampa nueva (la que falte).  
**K a explorar:** 1, 2, 5, 10.

In [ ]:
# ── Función de simulación con intercambio ────────────────────────────────────
def simular_con_intercambio(K, max_sobres=None, rng_sim=None):
    """
    Simula la compra de sobres con mecanismo de canje.

    K          : repetidas necesarias para canjear 1 nueva
    max_sobres : si se especifica, compra exactamente esa cantidad;
                 si es None, compra hasta completar el álbum.
    rng_sim    : generador numpy a usar.

    Retorna (completado: bool, sobres_usados: int)
    """
    coleccion  = set()                          # estampas ya obtenidas
    faltantes  = list(range(N))                 # estampas que aún faltan
    repetidas  = 0                              # acumulado de repetidas
    sobres     = 0

    while True:
        if max_sobres is None:
            if len(coleccion) == N:
                break
        else:
            if sobres >= max_sobres:
                break

        # Comprar un sobre
        stickers = rng_sim.integers(0, N, size=S)
        sobres  += 1

        for st in stickers:
            if st not in coleccion:
                coleccion.add(st)
                if st in faltantes_set:
                    faltantes_set.discard(st)
            else:
                repetidas += 1

        # Canjear repetidas si hay faltantes
        while repetidas >= K and faltantes_set:
            repetidas -= K
            nueva = rng_sim.choice(list(faltantes_set))
            coleccion.add(nueva)
            faltantes_set.discard(nueva)

    return len(coleccion) == N, sobres


# Versión corregida con faltantes_set como variable local
def sim_intercambio(K, max_sobres=None, rng_sim=None):
    coleccion   = set()
    faltantes_s = set(range(N))
    repetidas   = 0
    sobres      = 0

    while True:
        if max_sobres is None:
            if len(coleccion) == N:
                break
        else:
            if sobres >= max_sobres:
                break

        stickers = rng_sim.integers(0, N, size=S)
        sobres  += 1

        for st in stickers:
            if st not in coleccion:
                coleccion.add(st)
                faltantes_s.discard(st)
            else:
                repetidas += 1

        # Canje tras cada sobre
        while repetidas >= K and faltantes_s:
            repetidas -= K
            nueva = rng_sim.choice(list(faltantes_s))
            coleccion.add(nueva)
            faltantes_s.discard(nueva)

    return len(coleccion) == N, sobres

print("Función sim_intercambio definida correctamente.")

### Parte A: Simulación hasta completar el álbum para distintos K

In [ ]:
K_valores  = [1, 2, 5, 10]
resultados_A = {}   # K → array de sobres

# Caso base sin intercambio
rng_base = np.random.default_rng(2026)
sobres_base = []
for _ in range(R):
    coleccion = np.zeros(N, dtype=bool)
    s = 0
    while coleccion.sum() < N:
        coleccion[rng_base.integers(0, N, size=S)] = True
        s += 1
    sobres_base.append(s)
sobres_base = np.array(sobres_base)
media_base  = sobres_base.mean()

print(f"Sin intercambio — media: {media_base:.2f} sobres, std: {sobres_base.std():.2f}")
print()

for K in K_valores:
    rng_k = np.random.default_rng(2026)
    sobres_k = []
    for _ in range(R):
        _, s = sim_intercambio(K, max_sobres=None, rng_sim=rng_k)
        sobres_k.append(s)
    sobres_k = np.array(sobres_k)
    resultados_A[K] = sobres_k

    media_k  = sobres_k.mean()
    std_k    = sobres_k.std()
    reduccion = (media_base - media_k) / media_base * 100
    print(f"K={K:2d} → media: {media_k:.2f} sobres | std: {std_k:.2f} | "
          f"reducción vs base: {reduccion:.1f}%")

In [ ]:
# ── Histogramas superpuestos ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

colores_k = {1: "#e74c3c", 2: "#e67e22", 5: "#3498db", 10: "#9b59b6"}

ax.hist(sobres_base, bins=60, alpha=0.4, label="Sin intercambio",
        color="gray", density=True)

for K in K_valores:
    ax.hist(resultados_A[K], bins=60, alpha=0.4,
            label=f"K = {K}", color=colores_k[K], density=True)

ax.set_xlabel("Número de sobres hasta completar", fontsize=12)
ax.set_ylabel("Densidad", fontsize=12)
ax.set_title("Etapa 4A — Distribución del número de sobres por valor de K", fontsize=12)
ax.legend()
ax.grid(axis="y", linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("etapa4A_histogramas.png", dpi=150)
plt.show()

### Parte B: Probabilidad de éxito en función de M sobres para distintos K

In [ ]:
M_valores = [20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70]
resultados_B = {}   # K → lista de probabilidades (una por M)

# Caso base sin intercambio
probs_base = []
for M in M_valores:
    rng_bm = np.random.default_rng(2026)
    exitos = 0
    for _ in range(R):
        col = np.zeros(N, dtype=bool)
        for _ in range(M):
            col[rng_bm.integers(0, N, size=S)] = True
        if col.sum() == N:
            exitos += 1
    probs_base.append(exitos / R)

print("Calculando probabilidades para cada K y M...")
for K in K_valores:
    probs_k = []
    for M in M_valores:
        rng_km = np.random.default_rng(2026)
        exitos = 0
        for _ in range(R):
            completado_sim, _ = sim_intercambio(K, max_sobres=M, rng_sim=rng_km)
            if completado_sim:
                exitos += 1
        probs_k.append(exitos / R)
    resultados_B[K] = probs_k
    print(f"  K={K} listo.")

print("\nTabla de probabilidades:")
header = f"{'M':>4} | {'Sin K':>7} | " + " | ".join(f"K={k:2d}" for k in K_valores)
print(header)
print("-" * len(header))
for j, M in enumerate(M_valores):
    row = f"{M:4d} | {probs_base[j]:6.4f}  | "
    row += " | ".join(f"{resultados_B[k][j]:5.4f}" for k in K_valores)
    print(row)

In [ ]:
# ── Gráfica de líneas: probabilidad vs M ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(M_valores, probs_base, "o--", color="gray",  label="Sin intercambio", linewidth=1.5)

for K in K_valores:
    ax.plot(M_valores, resultados_B[K], "o-",
            color=colores_k[K], label=f"K = {K}", linewidth=1.8)

for nivel, estilo in [(0.50, ":"), (0.75, "-."), (0.90, "--")]:
    ax.axhline(nivel, color="black", linestyle=estilo, linewidth=0.8, alpha=0.6)
    ax.text(M_valores[-1] + 0.3, nivel, f"{int(nivel*100)}%",
            va="center", fontsize=8)

ax.set_xlabel("Número de sobres M", fontsize=12)
ax.set_ylabel("Probabilidad de completar", fontsize=12)
ax.set_title("Etapa 4B — P(completar) en función de M para distintos K", fontsize=12)
ax.set_xticks(M_valores)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend(loc="upper left")
ax.grid(linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("etapa4B_lineas.png", dpi=150)
plt.show()

In [ ]:
# ── M necesario para alcanzar 50%, 75% y 90% de probabilidad (interpolación) ─
from numpy import interp

niveles = [0.50, 0.75, 0.90]

def m_para_nivel(probs, nivel):
    """Interpola linealmente el M necesario para alcanzar `nivel` de probabilidad."""
    probs_arr = np.array(probs)
    M_arr     = np.array(M_valores)
    if probs_arr[-1] < nivel:
        return f"> {M_arr[-1]}"
    if probs_arr[0] >= nivel:
        return f"< {M_arr[0]}"
    # interp requiere x creciente; las probabilidades deben ser crecientes en M
    return f"{interp(nivel, probs_arr, M_arr):.1f}"

print(f"{'K':>6} | {'50%':>7} | {'75%':>7} | {'90%':>7}")
print("-" * 35)

print(f"{'Sin K':>6} | "
      + " | ".join(f"{m_para_nivel(probs_base, n):>7}" for n in niveles))

for K in K_valores:
    row = f"{K:>6} | "
    row += " | ".join(f"{m_para_nivel(resultados_B[K], n):>7}" for n in niveles)
    print(row)

### Preguntas de análisis — Etapa 4

In [ ]:
# ── Pregunta 2: ahorro de K=2 respecto a sin intercambio ────────────────────
ahorro_sobres = media_base - resultados_A[2].mean()
ahorro_Q      = ahorro_sobres * PRECIO

print("=" * 55)
print("Pregunta 2 — Ahorro con K=2")
print("=" * 55)
print(f"Media sin intercambio:  {media_base:.2f} sobres")
print(f"Media con K=2:          {resultados_A[2].mean():.2f} sobres")
print(f"Ahorro promedio:        {ahorro_sobres:.2f} sobres")
print(f"Ahorro en quetzales:    Q {ahorro_Q:.2f}")

In [ ]:
# ── Pregunta 3: comparación en M=45 ─────────────────────────────────────────
M_idx = M_valores.index(45)

p_sinK  = probs_base[M_idx]
p_K10   = resultados_B[10][M_idx]
p_K5    = resultados_B[5][M_idx]
p_K1    = resultados_B[1][M_idx]

print("=" * 55)
print("Pregunta 3 — Probabilidades en M = 45")
print("=" * 55)
print(f"P sin intercambio: {p_sinK:.4f}")
print(f"P con K=10:        {p_K10:.4f}")
print(f"P con K=5:         {p_K5:.4f}")
print(f"P con K=1:         {p_K1:.4f}")
print()
print(f"Aumento K=10 → K=5: {(p_K5 - p_K10)*100:+.2f} pp")
print(f"Aumento K=5  → K=1: {(p_K1 - p_K5)*100:+.2f} pp")

In [ ]:
# ── Pregunta 4: exploración de valores adicionales de K ─────────────────────
K_extra = [1, 2, 3, 4, 5, 7, 10, 15, 20]
medias_extra = {}

for K in K_extra:
    rng_e = np.random.default_rng(2026)
    s_list = [sim_intercambio(K, max_sobres=None, rng_sim=rng_e)[1] for _ in range(R)]
    medias_extra[K] = np.mean(s_list)

print("Exploración de K adicionales (media de sobres hasta completar):")
print(f"{'K':>4} | {'Media sobres':>14} | {'Reducción vs base':>18}")
print("-" * 42)
for K, med in medias_extra.items():
    red = (media_base - med) / media_base * 100
    print(f"{K:4d} | {med:14.2f} | {red:17.1f}%")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(list(medias_extra.keys()), list(medias_extra.values()), "o-", color="steelblue")
ax.axhline(media_base, linestyle="--", color="gray", label="Sin intercambio")
ax.set_xlabel("K (repetidas por canje)", fontsize=11)
ax.set_ylabel("Media de sobres", fontsize=11)
ax.set_title("Pregunta 4 — Media de sobres vs K", fontsize=11)
ax.legend()
ax.grid(linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("etapa4_pregunta4.png", dpi=150)
plt.show()

In [ ]:
# ── Pregunta 5: costo efectivo por estampa nueva mediante canje ──────────────
print("=" * 55)
print("Pregunta 5 — Costo efectivo por estampa nueva via canje")
print("=" * 55)
print()
print("Cada sobre contiene S=7 estampas y cuesta Q 9.50.")
print("Costo por estampa de sobre: Q 9.50/7 ≈ Q {:.4f}".format(PRECIO/S))
print()
print("Con canje, K repetidas (ya pagadas) dan 1 nueva.")
print("El costo de obtener K repetidas depende del proceso,")
print("pero cada repetida 'costó' Q {:.4f} cuando se compró.".format(PRECIO/S))
print()
for K in [1, 2, 5, 10]:
    costo_canje = K * (PRECIO / S)
    print(f"  K={K:2d}: costo efectivo por estampa nueva = Q {costo_canje:.4f}")
print()
print("A menor K, menor costo efectivo por estampa vía canje.")
print("K=1 es la más rentable (1 repetida → 1 nueva).")